## Overview
As an automotive supplier, we are interested in understanding the broader trends in the automotive
industry. For this task, you will create a data pipeline to acquire, process, and load data from various
public sources.

## Objective & Tasks
Your objective is to create a script or a series of scripts (preferably in Python or SQL) that:
**1. Data Acquisition:** Write scripts to download the datasets from the provided public data sources

**2. Data Processing:** Clean and integrate these datasets. This should include, but not be limited to,
handling missing values, duplicates, and possible outliers.

**3. Data Transformation:** Transform the data into a format suitable for further analysis. Justify the
choices you make during this process.

**4. Data Loading:** Write a script to load the data into a hypothetical data storage system. While you
cannot actually load the data into Azure SQL Database or Databricks Delta Lake, you should
simulate the process and include the relevant commands in your script.

**5. Automation Suggestion:** Describe how you would automate this pipeline with a schedule interval
you would choose and explain why.

# Use the following data sources for this task:

U.S. Department of Transportation - National Highway Traffic Safety Administration: Vehicle
Complaints https://afdc.energy.gov/data_download 


# Datacard

The Alternative Fuels Data Center (AFDC) provides information, data, and tools to help fleets, fuel providers, policymakers, cities, states, Clean Cities and Communities coalitions, and other transportation decision makers find ways to reach their energy, environmental, and economic goals through the use of alternative and renewable fuels, advanced vehicles, and other fuel-saving strategies.

Data Included in the Alternative Fuel Stations Download https://afdc.energy.gov/data_download/alt_fuel_stations_format 

##Step 1: Download ZIP file

Download csv file from https://afdc.energy.gov/data_download with historical data from the Station Locator starting in 2014 until present.

For future incremental load just use snapshot of data in the Station Locator today option to download current data.

## Step 2: Create DBFS folder for RAW CSV

Created separate folder in DBFS to store the raw csv

Move the CSV to correct path

Verify the file path

In [0]:
dbutils.fs.mkdirs("/mnt/afdc/raw")

In [0]:
dbutils.fs.mv(
    "dbfs:/mnt/alt_fuel_stations_historical_day__Jan_25_2021_.csv"
    "dbfs:/mnt/afdc/raw/",
    recurse=True
)

In [0]:
display(dbutils.fs.ls("dbfs:/mnt/afdc/raw/"))

## Step 3: Read the data from the csv raw

In [0]:
df_bronze = spark.read.csv(
    "dbfs:/mnt/afdc/raw/alt_fuel_stations_historical_day__Jan_25_2021_.csv",
    header=True,
    inferSchema=True
)

df_bronze.printSchema()

## Step 4: Add a new column Data_update_timestamp dynamically.

This column will store the current timestamp each time the pipeline runs (including incremental updates).

It automatically captures the current system timestamp when the pipeline runs. Works for full load and incremental updates.

In [0]:
from pyspark.sql.functions import current_timestamp

# Add timestamp column
df_bronze = df_bronze.withColumn("Data_update_timestamp", current_timestamp())

## Step 5: Write the DataFrame to Bronze Delta Table

Replaces all invalid characters in column names with underscores, allowing you to save the DataFrame as a Delta table without errors.



In [0]:
invalid_chars = [' ', ',', ';', '{', '}', '(', ')', '\n', '\t', '=']
def clean_col(col_name):
    for ch in invalid_chars:
        col_name = col_name.replace(ch, '_')
    return col_name

df_bronze = df_bronze.toDF(*[clean_col(c) for c in df_bronze.columns])

bronze_path = "dbfs:/mnt/afdc/bronze"
df_bronze.write.format("delta").mode("overwrite").option("mergeSchema", "true").save(bronze_path)

print(f"Bronze table saved at {bronze_path}")

## Step 6: Register the bronze table in the metastore for SQL access



In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS adfc_alternative_fuel_stations.bronze
USING DELTA
LOCATION '{bronze_path}'
""")

## Step 7: Display the bronze table

In [0]:
spark.read.table("hive_metastore.adfc_alternative_fuel_stations.bronze").display()